In [6]:
import os
import sys
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate
from huggingface_hub import ModelCard, ModelCardData

# ==========================================
# STEP 1: CARREGAR MODELO E DADOS DO EDA
# ==========================================
print("\n⏳ Carregando modelo treinado e dados pré-processados...")

# Carregar o modelo baseline do EDA
model = joblib.load("../models/baseline_model.joblib")
print("✓ Modelo Logistic Regression carregado")

# Carregar o dataframe pré-processado
df_full = pd.read_csv("../data/raw/Telco_customer_churn_preprocessed.csv")
print(f"✓ Dataset carregado com shape: {df_full.shape}")

# Preparar os dados para treino e teste
X = df_full.drop(columns=["target"])
y = df_full["target"]

# Replicar o mesmo split usado no EDA
RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

print(f"✓ Dados separados em treino ({X_train.shape[0]}) e teste ({X_test.shape[0]})")

# ==========================================
# STEP 2: GERAR PREDIÇÕES E MÉTRICAS
# ==========================================
print("\n📊 Gerando predições e calculando métricas...")

# Predições
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)
y_prob_test = model.predict_proba(X_test)[:, 1]

# Métricas de Performance
metrics_dict = {
    "Acurácia (Treino)": accuracy_score(y_train, y_pred_train),
    "Acurácia (Teste)": accuracy_score(y_test, y_pred_test),
    "Precisão": precision_score(y_test, y_pred_test),
    "Recall": recall_score(y_test, y_pred_test),
    "F1-Score": f1_score(y_test, y_pred_test),
    "ROC-AUC": roc_auc_score(y_test, y_prob_test),
}

print("\n=== MÉTRICAS DE PERFORMANCE ===")
for metric_name, metric_value in metrics_dict.items():
    print(f"{metric_name}: {metric_value:.4f}")

# ==========================================
# STEP 3: AUDITORIA DE JUSTIÇA COM FAIRLEARN
# ==========================================
print("\n🔍 Executando auditoria de justiça (Fairlearn)...")

# Carregar o dataset original para extrair atributos sensíveis
df_original = pd.read_excel("../data/raw/Telco_customer_churn.xlsx")
df_original.columns = [c.lower().replace(" ", "_") for c in df_original.columns]

# Usar gênero como atributo sensível
sensitive_feature = df_original["gender"].values

# Garantir que temos os mesmos índices para o split
_, _, _, _, grupo_train, grupo_test = train_test_split(
    X, y, sensitive_feature, test_size=0.2, random_state=RANDOM_STATE
)

# Avaliar fairness usando MetricFrame
metric_frame = MetricFrame(
    metrics={
        "Acurácia": accuracy_score,
        "Precisão": precision_score,
        "Taxa de Seleção (Churn)": selection_rate,
        "Taxa de Falsos Positivos": false_positive_rate
    },
    y_true=y_test,
    y_pred=y_pred_test,
    sensitive_features=grupo_test
)

# Tabela de métricas por grupo
df_fairness = metric_frame.by_group.round(4)
tabela_markdown_fairness = df_fairness.to_markdown()

# Calcular disparidades
disparidades = metric_frame.difference()
diff_acuracia = disparidades["Acurácia"]
diff_precisao = disparidades["Precisão"]
diff_selecao = disparidades["Taxa de Seleção (Churn)"]

print("\n=== ANÁLISE DE DISPARIDADE (Fairness) ===")
print(f"Disparidade de Acurácia: {diff_acuracia:.4f}")
print(f"Disparidade de Precisão: {diff_precisao:.4f}")
print(f"Disparidade de Taxa de Seleção: {diff_selecao:.4f}")

print("\n" + tabela_markdown_fairness)

# ==========================================
# STEP 4: CONSTRUÇÃO DO MODEL CARD
# ==========================================
print("\n📝 Gerando Model Card...")

model_version = "1.0.0"
model_date = "Maio 2026"
release_notes = """Release inicial com Logistic Regression"""

# Metadados estruturados
card_metadata = ModelCardData(
    language="pt",
    license="mit",
    library_name="sklearn",
    tags=["telco-churn", "customer-retention", "fairness-audit", "governance"],
    metrics=["accuracy", "precision", "recall", "f1", "roc-auc"],
)

# Conteúdo do Model Card
content_template = f"""
# Modelo de Previsão de Churn (Tech Challenge - Fase 1)

## Sumário

Este Model Card documenta o modelo preditivo de cancelamento de clientes (Churn). O modelo utiliza uma abordagem de classificação binária com regressão logística, balanceada para lidar com o desbalanceamento de classes presente nos dados históricos.

---

## 1. Detalhes do Modelo

| Atributo | Valor |
|----------|-------|
| **Nome** | Baseline Churn Prediction Model |
| **Tipo** | Classificação Binária |
| **Algoritmo** | Regressão Logística |
| **Framework** | scikit-learn |
| **Versão do Modelo** | 1.0.0 |
| **Data de Criação** | Maio 2026 |
| **Licença** | MIT |
| **Responsável** | Equipe de MLEks - Tech Challenge |

---

## 2. Uso Pretendido

### Casos de Uso Recomendados
- Identificação de clientes em risco de cancelamento para ações proativas de retenção
- Segmentação de clientes por probabilidade de churn
- Análise de padrões e fatores de risco associados ao cancelamento
- Suporte a decisões estratégicas de retenção e pricing

### Fora de Escopo
- O modelo não deve ser usado de forma autônoma sem supervisão humana
- Não recomendado para decisões disciplinares ou penalizações
- Requer reavaliação periódica em cenários de mudanças de mercado

---

## 3. Dados de Treinamento e Validação

### Características do Dataset
- **Fonte Principal**: [Telco customer churn: IBM dataset](https://www.kaggle.com/datasets/yeanzc/telco-customer-churn-ibm-dataset)
- **Tamanho Original**: 7.043 registros
- **Período de Cobertura**: Dados históricos de clientes de telecomunicações
- **Divisão**: 80% Treino / 20% Teste

### Features Utilizadas
- **Dados Demográficos**: Idade, Gênero, Parceria, Dependentes
- **Dados de Contrato**: Tipo de Contrato, Duração da Relação (tenure)
- **Dados de Serviço**: Internet, Telefonia, Segurança Online, Backup, etc.
- **Dados Financeiros**: Cobrança Mensal, Gasto Médio Mensal, Método de Pagamento

### Atributos Sensíveis Mapeados
- **Gênero (Masculino/Feminino)** - Utilizado exclusivamente para auditoria pós-treino
- **Senior Citizen** - Monitorado para vieses etários

---

## 4. Análise de Performance

### Métricas Agregadas
| Métrica | Valor |
|---------|-------|
| **Acurácia (Treino)** | {metrics_dict['Acurácia (Treino)']:.4f} |
| **Acurácia (Teste)** | {metrics_dict['Acurácia (Teste)']:.4f} |
| **Precisão** | {metrics_dict['Precisão']:.4f} |
| **Recall (Sensibilidade)** | {metrics_dict['Recall']:.4f} |
| **F1-Score** | {metrics_dict['F1-Score']:.4f} |
| **ROC-AUC** | {metrics_dict['ROC-AUC']:.4f} |

### Interpretação
- **Precisão {metrics_dict['Precisão']:.2%}**: Do total de predições positivas (churn), ~{metrics_dict['Precisão']*100:.0f}% estão corretas
- **Recall {metrics_dict['Recall']:.2%}**: O modelo identifica ~{metrics_dict['Recall']*100:.0f}% dos clientes que realmente vão fazer churn
- **F1-Score {metrics_dict['F1-Score']:.4f}**: Balanço entre precisão e recall
- **ROC-AUC {metrics_dict['ROC-AUC']:.4f}**: Capacidade discriminativa do modelo

---

## 5. Análise de Equidade (Fairness)

### Métricas Desagregadas por Gênero

{tabela_markdown_fairness}

### Avaliação de Disparidade
| Métrica de Disparidade | Valor | Interpretação |
|------------------------|-------|----------------|
| Disparidade de Acurácia | {diff_acuracia:.4f} | Diferença máxima de acurácia entre grupos |
| Disparidade de Precisão | {diff_precisao:.4f} | Diferença máxima de precisão entre grupos |
| Disparidade de Taxa de Seleção | {diff_selecao:.4f} | Diferença na proporção de previsões positivas |

### Conclusões sobre Equidade
✓ **Status**: Modelo demonstra equidade razoável entre grupos demográficos
- Disparidades detectadas estão dentro de limites aceitáveis (<0.10)
- Não há viés sistemático evidente contra nenhum grupo
- Recomenda-se monitoramento contínuo durante operacionalização

---

## 6. Considerações Éticas e Limitações

### Potenciais Vieses
1. **Viés de Dados Históricos**: O modelo herda padrões dos dados históricos
2. **Desbalanceamento de Classes**: ~26% de churn vs 74% de retenção
3. **Mudanças de Mercado**: Performance pode degradar com mudanças econômicas

### Estratégias de Mitigação
- ✓ Uso de `class_weight='balanced'` para compensar desbalanceamento
- ✓ Auditoria periódica com Fairlearn
- ✓ Monitoramento de drift em produção
- ✓ Reavaliação a cada 3 meses com dados novos

### Recomendações
- Combinar com julgamento humano antes de tomar decisões
- Investigar razões por trás de previsões de alto risco
- Manter registro de feedback de especialistas em negócio
- Estabelecer limites de confiança (thresholds) para ação

---

## 7. Como Usar o Modelo em Produção

```python
import joblib
import pandas as pd

# Carregar modelo
model = joblib.load("baseline_model.joblib")

# Preparar dados novos (mesma estrutura do treinamento)
# X_novo deve ter mesmas features do treinamento
y_pred = model.predict(X_novo)
y_prob = model.predict_proba(X_novo)[:, 1]

# y_pred: 0 (sem churn) ou 1 (com churn)
# y_prob: probabilidade de churn [0-1]
```

---

## 8. Versionamento e Histórico

| Versão | Data | Principais Mudanças |
|--------|------|---------------------|
| {model_version} | {model_date} | {release_notes} |

---

## 9. Contato e Suporte

- **Equipe Responsável**: MLEks - Tech Challenge
- **Para Dúvidas**: Consultar documentação no repositório
- **Avisos de Retraining**: Monitorar performance mensal
"""

# Criar e salvar Model Card
card = ModelCard.from_template(card_data=card_metadata, template_str=content_template)
output_path = "../docs/docs/MODEL_CARD.md"
card.save(output_path)

print(f"\n✅ Model Card gerado com sucesso!")
print(f"📄 Arquivo salvo em: {os.path.abspath(output_path)}")



⏳ Carregando modelo treinado e dados pré-processados...
✓ Modelo Logistic Regression carregado
✓ Dataset carregado com shape: (7043, 31)
✓ Dados separados em treino (5634) e teste (1409)

📊 Gerando predições e calculando métricas...

=== MÉTRICAS DE PERFORMANCE ===
Acurácia (Treino): 0.7419
Acurácia (Teste): 0.7466
Precisão: 0.5354
Recall: 0.8125
F1-Score: 0.6455
ROC-AUC: 0.8463

🔍 Executando auditoria de justiça (Fairlearn)...


Repo card metadata block was not found. Setting CardData to empty.



=== ANÁLISE DE DISPARIDADE (Fairness) ===
Disparidade de Acurácia: 0.0017
Disparidade de Precisão: 0.0211
Disparidade de Taxa de Seleção: 0.0299

| sensitive_feature_0   |   Acurácia |   Precisão |   Taxa de Seleção (Churn) |   Taxa de Falsos Positivos |
|:----------------------|-----------:|-----------:|--------------------------:|---------------------------:|
| Female                |     0.7458 |     0.5455 |                    0.4455 |                     0.2871 |
| Male                  |     0.7475 |     0.5243 |                    0.4156 |                     0.2718 |

📝 Gerando Model Card...

✅ Model Card gerado com sucesso!
📄 Arquivo salvo em: /Users/dionatasantosdasilva/dev/repos/personal/postech-ml/tech-challenge-fase-1/docs/docs/MODEL_CARD.md
